# 02.4 — Query Expansion

**Problem it solves:** Users query with 'car', documents say 'automobile'. BM25 scores zero. Query expansion adds synonyms, related terms, or feedback documents to bridge this vocabulary gap.

**Main approaches:**
1. **Thesaurus-based**: WordNet synonyms added to query
2. **Pseudo-Relevance Feedback (PRF)**: Take top-k results, extract key terms, re-query
3. **Word embedding expansion**: Use Word2Vec/GloVe nearest neighbors

---

In [ ]:
import re, math
from collections import Counter

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())

# ---- reuse BM25 from previous notebook ----
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1=k1; self.b=b
        self.docs=[]; self.doc_freqs=[]; self.doc_lens=[]
        self.df=Counter(); self.idf={}; self.avgdl=0; self.corpus_size=0
    def fit(self, docs):
        self.docs=docs; self.corpus_size=len(docs)
        for doc in docs:
            toks=tokenize(doc); self.doc_freqs.append(Counter(toks))
            self.doc_lens.append(len(toks)); self.df.update(set(toks))
        self.avgdl=sum(self.doc_lens)/self.corpus_size
        N=self.corpus_size
        self.idf={t: math.log((N-f+0.5)/(f+0.5)+1) for t,f in self.df.items()}
        return self
    def score(self, query, doc_id):
        s=0.0; dl=self.doc_lens[doc_id]; tf_d=self.doc_freqs[doc_id]
        for t in tokenize(query):
            if t not in self.idf: continue
            tf=tf_d.get(t,0)
            s+=self.idf[t]*tf*(self.k1+1)/(tf+self.k1*(1-self.b+self.b*dl/self.avgdl))
        return s
    def search(self, query, top_k=5):
        scores=sorted([(self.score(query,i),i) for i in range(self.corpus_size)],reverse=True)
        return [(s,i,self.docs[i]) for s,i in scores[:top_k] if s>0]

docs = [
    "automobile manufacturers face supply chain challenges",
    "car sales declined in the fourth quarter",
    "electric vehicle adoption is accelerating globally",
    "the automobile industry is transitioning to electric cars",
    "fuel economy standards affect vehicle design",
    "natural language processing requires large text corpora",
    "machine learning models need training data",
    "self-driving vehicles use computer vision and sensors",
]
bm25 = BM25().fit(docs)

print("Original query: 'car'")
for s,i,d in bm25.search('car'):
    print(f"  [{i}] {s:.3f}  {d}")

In [ ]:
# === APPROACH 1: Thesaurus-Based Expansion ===

# A mini thesaurus (in production: WordNet or domain-specific)
THESAURUS = {
    'car':        ['automobile', 'vehicle', 'auto'],
    'automobile': ['car', 'vehicle', 'auto'],
    'fast':       ['quick', 'rapid', 'speedy', 'swift'],
    'big':        ['large', 'huge', 'enormous', 'massive'],
    'small':      ['tiny', 'little', 'miniature', 'compact'],
    'buy':        ['purchase', 'acquire', 'obtain'],
    'text':       ['document', 'corpus', 'content', 'prose'],
}

def thesaurus_expand(query: str, thesaurus: dict, 
                     max_expansions: int = 3) -> str:
    """Expand each query term with its synonyms."""
    terms = tokenize(query)
    expanded = list(terms)
    for term in terms:
        if term in thesaurus:
            synonyms = thesaurus[term][:max_expansions]
            expanded.extend(synonyms)
    return ' '.join(expanded)

original_query = 'car'
expanded_query = thesaurus_expand(original_query, THESAURUS)
print(f"Original:  {original_query!r}")
print(f"Expanded:  {expanded_query!r}")
print()
print("Results after thesaurus expansion:")
for s,i,d in bm25.search(expanded_query):
    print(f"  [{i}] {s:.3f}  {d}")

In [ ]:
# === APPROACH 2: Pseudo-Relevance Feedback (PRF) ===
# Assume top-k results are relevant.
# Extract the most distinctive terms from those documents.
# Re-query with original + top extracted terms.
# Also called: Rocchio algorithm, RM3 (in dense retrieval)

def pseudo_relevance_feedback(query: str, bm25: BM25, 
                               top_k: int = 3, 
                               n_expand_terms: int = 5) -> str:
    """
    Expand query using terms from the top-k results.
    Pick terms with highest TF-IDF score in the pseudo-relevant set.
    """
    # Step 1: initial retrieval
    top_results = bm25.search(query, top_k=top_k)
    if not top_results:
        return query
    
    # Step 2: collect terms from top results
    feedback_doc_ids = [i for _, i, _ in top_results]
    
    # Step 3: score each term by how well it describes the pseudo-relevant set
    # Use a simplified RM3-style weighting: TF in feedback docs × IDF in corpus
    term_scores: Counter = Counter()
    query_terms = set(tokenize(query))
    
    for doc_id in feedback_doc_ids:
        for term, tf in bm25.doc_freqs[doc_id].items():
            if term in query_terms:
                continue  # already in query
            idf = bm25.idf.get(term, 0)
            doc_len = bm25.doc_lens[doc_id]
            # Normalized TF × IDF
            term_scores[term] += (tf / doc_len) * idf
    
    # Step 4: take top n_expand_terms
    expansion_terms = [term for term, _ in term_scores.most_common(n_expand_terms)]
    expanded = query + ' ' + ' '.join(expansion_terms)
    
    return expanded, expansion_terms

query = 'car'
expanded, new_terms = pseudo_relevance_feedback(query, bm25)
print(f"Original query: {query!r}")
print(f"PRF added terms: {new_terms}")
print(f"Expanded query: {expanded!r}")
print()
print("Results after PRF:")
for s,i,d in bm25.search(expanded):
    print(f"  [{i}] {s:.3f}  {d}")

In [ ]:
# PRF failure case: query drift
# If initial results are bad, PRF makes things worse

bad_query = 'learning'  # ambiguous — could be ML or general
print(f"\nQuery: {bad_query!r}")
print("Initial results (potentially off-topic):")
for s,i,d in bm25.search(bad_query):
    print(f"  [{i}] {s:.3f}  {d}")

expanded, new_terms = pseudo_relevance_feedback(bad_query, bm25)
print(f"\nPRF adds: {new_terms}")
print("After PRF (topic may have drifted):")
for s,i,d in bm25.search(expanded):
    print(f"  [{i}] {s:.3f}  {d}")
print("\nIf top results were off-topic, PRF amplifies the error.")

In [ ]:
# === APPROACH 3: When Expansion Hurts ===

# Query: 'Java' (programming language)
# Thesaurus expansion: 'Java coffee island' — WRONG direction

# Query: 'Turkey' (the country)
# Expansion might add 'bird food thanksgiving' — WRONG

# The polysemy problem:
ambiguous_queries = {
    'java':    ['a programming language', 'a type of coffee', 'an Indonesian island'],
    'turkey':  ['a country in Asia', 'a large bird', 'a poor performance in bowling'],
    'bank':    ['a financial institution', 'the side of a river', 'to tilt an aircraft'],
    'python':  ['a programming language', 'a large snake', 'a Monty Python reference'],
}

print("Polysemous words — thesaurus expansion picks ONE sense, often wrong:")
for word, senses in ambiguous_queries.items():
    print(f"  {word:10} could mean: {' | '.join(senses)}")

print()
print("Takeaway: expansion is only safe when query intent is clear.")
print("Modern approach: use LLM to generate hypothetical answer document (HyDE),")
print("then embed that for dense retrieval — no explicit expansion needed.")

## Summary

**Query expansion improves recall at the cost of precision.**

| Method | Recall | Precision | When to use |
|--------|--------|-----------|-------------|
| Thesaurus | ↑ | ↓↓ | Controlled vocab, no polysemy |
| PRF | ↑ | ↓ | Good initial results, large corpus |
| Neural (HyDE) | ↑↑ | ↓ | Modern systems with LLM access |

**What breaks it:**
- Polysemy: expansion picks the wrong word sense
- Query drift in PRF: bad top-k → worse expansion
- Short queries expand into noise

---
**Next:** 02.5 — Evaluation: precision, recall, NDCG